In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

for f in sorted(RAW.glob("*.csv")):
    size_mb = f.stat().st_size / 1_000_000
    print(f"{f.name:<45} {size_mb:6.1f} MB")

olist_customers_dataset.csv                      9.0 MB
olist_geolocation_dataset.csv                   61.3 MB
olist_order_items_dataset.csv                   15.4 MB
olist_order_payments_dataset.csv                 5.8 MB
olist_order_reviews_dataset.csv                 14.5 MB
olist_orders_dataset.csv                        17.7 MB
olist_products_dataset.csv                       2.4 MB
olist_sellers_dataset.csv                        0.2 MB
product_category_name_translation.csv            0.0 MB


In [2]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

orders       = pd.read_csv(RAW / "olist_orders_dataset.csv")
order_items  = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments     = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers    = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers      = pd.read_csv(RAW / "olist_sellers_dataset.csv")
products     = pd.read_csv(RAW / "olist_products_dataset.csv")
translation  = pd.read_csv(RAW / "product_category_name_translation.csv")
geolocation  = pd.read_csv(RAW / "olist_geolocation_dataset.csv")

tables = {
    "orders": orders, "order_items": order_items, "payments": payments,
    "reviews": reviews, "customers": customers, "sellers": sellers,
    "products": products, "translation": translation, "geolocation": geolocation,
}
print("Loaded", len(tables), "tables")

Loaded 9 tables


In [3]:
summary = pd.DataFrame({
    "rows":    {name: df.shape[0] for name, df in tables.items()},
    "columns": {name: df.shape[1] for name, df in tables.items()},
})
summary

,rows,columns
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
sellers,3095,4
products,32951,9
translation,71,2
geolocation,1000163,5


In [4]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.dtypes.to_string())


=== orders ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

=== order_items ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

=== payments ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

=== reviews ===
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str

=== customers ===
custo

In [5]:
def null_report(df):
    out = pd.DataFrame({
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
    })
    return out[out["nulls"] > 0]

for name, df in tables.items():
    print(f"\n=== {name} ===")
    report = null_report(df)
    print(report.to_string() if len(report) else "no nulls")


=== orders ===
                               nulls  null_pct
order_approved_at                160      0.16
order_delivered_carrier_date    1783      1.79
order_delivered_customer_date   2965      2.98

=== order_items ===
no nulls

=== payments ===
no nulls

=== reviews ===
                        nulls  null_pct
review_comment_title    87656     88.34
review_comment_message  58247     58.70

=== customers ===
no nulls

=== sellers ===
no nulls

=== products ===
                            nulls  null_pct
product_category_name         610      1.85
product_name_lenght           610      1.85
product_description_lenght    610      1.85
product_photos_qty            610      1.85
product_weight_g                2      0.01
product_length_cm               2      0.01
product_height_cm               2      0.01
product_width_cm                2      0.01

=== translation ===
no nulls

=== geolocation ===
no nulls


In [6]:
pk_checks = {
    "orders":      ["order_id"],
    "customers":   ["customer_id"],
    "sellers":     ["seller_id"],
    "products":    ["product_id"],
    "translation": ["product_category_name"],
    "order_items": ["order_id", "order_item_id"],
    "payments":    ["order_id", "payment_sequential"],
    "reviews":     ["review_id"],
}

rows = []
for name, cols in pk_checks.items():
    df = tables[name]
    dup_rows = df.duplicated(subset=cols).sum()
    rows.append({"table": name, "key": cols, "rows": len(df), "duplicate_key_rows": dup_rows})

pd.DataFrame(rows)

,table,key,rows,duplicate_key_rows
0,orders,[order_id],99441,0
1,customers,[customer_id],99441,0
2,sellers,[seller_id],3095,0
3,products,[product_id],32951,0
4,translation,[product_category_name],71,0
5,order_items,"[order_id, order_item_id]",112650,0
6,payments,"[order_id, payment_sequential]",103886,0
7,reviews,[review_id],99224,814


In [7]:
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

for name, cols in date_cols.items():
    for c in cols:
        s = pd.to_datetime(tables[name][c], errors="coerce")
        print(f"{name}.{c:<35} min={s.min()}   max={s.max()}")

orders.order_purchase_timestamp            min=2016-09-04 21:15:19   max=2018-10-17 17:30:18
orders.order_approved_at                   min=2016-09-15 12:16:38   max=2018-09-03 17:40:06
orders.order_delivered_carrier_date        min=2016-10-08 10:34:01   max=2018-09-11 19:48:28
orders.order_delivered_customer_date       min=2016-10-11 13:46:32   max=2018-10-17 13:22:46
orders.order_estimated_delivery_date       min=2016-09-30 00:00:00   max=2018-11-12 00:00:00
order_items.shipping_limit_date                 min=2016-09-19 00:15:34   max=2020-04-09 22:35:08
reviews.review_creation_date                min=2016-10-02 00:00:00   max=2018-08-31 00:00:00
reviews.review_answer_timestamp             min=2016-10-07 18:32:28   max=2018-10-29 12:27:35


In [8]:
print(orders["order_status"].value_counts(dropna=False), "\n")
print(payments["payment_type"].value_counts(dropna=False), "\n")
print(reviews["review_score"].value_counts(dropna=False).sort_index(), "\n")

print("distinct customer states:", customers["customer_state"].nunique())
print("distinct seller states:  ", sellers["seller_state"].nunique())
print("distinct product categories:", products["product_category_name"].nunique())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64 

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64 

distinct customer states: 27
distinct seller states:   23
distinct product categories: 73


In [9]:
checks = {
    "order_items.price <= 0":          (order_items["price"] <= 0).sum(),
    "order_items.freight_value < 0":   (order_items["freight_value"] < 0).sum(),
    "payments.payment_value <= 0":     (payments["payment_value"] <= 0).sum(),
    "payments.payment_installments=0": (payments["payment_installments"] == 0).sum(),
    "products.weight_g <= 0":          (products["product_weight_g"] <= 0).sum(),
    "products.length/height/width <= 0": (
        (products[["product_length_cm", "product_height_cm", "product_width_cm"]] <= 0).any(axis=1).sum()
    ),
    "reviews.score not in 1..5":       (~reviews["review_score"].between(1, 5)).sum(),
}
pd.Series(checks, name="rows_flagged")

order_items.price <= 0               0
order_items.freight_value < 0        0
payments.payment_value <= 0          9
payments.payment_installments=0      2
products.weight_g <= 0               4
products.length/height/width <= 0    0
reviews.score not in 1..5            0
Name: rows_flagged, dtype: int64